In [ ]:
# UCell signature scoring and decoupleR TF ranking for both atlases.
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

In [ ]:
input_dir = "/home/yanjun3/atlas/heart_sc"

In [ ]:
import seaborn as sns
import torch

In [ ]:
sc.set_figure_params(figsize=(3, 3),  facecolor="white",
    frameon=True,   dpi = 300, format = 'png')
sc.settings.figdir = "/home/yanjun3/atlas/heart_sn/figures_scoring"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
adata = sc.read('/home/yanjun2/atlas_project/heart_sc_h5ad/qc3_scvi.h5ad')
adata1 = sc.read('/home/yanjun2/atlas_project/heart_sn_h5ad/sn_final.h5ad')

In [ ]:
sc_signatures = {

    # =========================
    # IMMUNITY & INFLAMMATION
    # =========================

    "leukocyte_activation_regulation": [
        "AIF1","LST1","TYROBP","FCER1G","CSF1R",
        "TNFAIP3","NFKBIA","NFKBIZ","ZC3H12A",
        "TRAF3","TRAF4","TAB2","TANK","OTULIN","BIRC3",
        "SIGIRR","VSIR"
    ],

    "innate_sensing_inflammasome": [
        "TLR4","TLR5","TLR8","UNC93B1",
        "NLRP3","PYCARD",
        "IRAK1","IRAK2","RIPK1","RIPK2","CARD8"
    ],

    "antigen_presentation": [
        "CD74","B2M","TAPBP",
        "CD40","CD80","CD83","CD58"
    ],

    "cytokine_secretion": [
        "TNF","IL1A","IL1B","IL6","IL10","IL18",
        "OSM","TNFSF13B","TNFSF15",
        "TNFRSF1B","TNFRSF12A"
    ],

    "chemotaxis_migration": [
        "CX3CR1","CXCR4",
        "CXCL1","CXCL2","CXCL3","CXCL8",
        "FPR3",
        "ITGAM","ITGAX","ITGB2",
        "ITGA5","ITGA6","ITGAV","ITGB1","ITGB8"
    ],

    "complement_coagulation": [
        "CR1","F3","SERPING1"
    ],

    # =========================
    # ENDOCYTOSIS & CLEARANCE
    # =========================

    "endocytosis_phagocytosis_clearance": [
        "LYZ","CD14","CD36","LRP1","SCARB2",
        "MARCO","MERTK","STAB1","VSIG4",
        "LAMP1","LAPTM5",
        "CTSS","CTSD","CTSL","CTSC","LGMN","ACP5",
        "RUBCN","SWAP70","WIPF1","CORO1A","RAC1","CDC42","RHOA"
    ],

    # =========================
    # ECM REMODELLING
    # =========================

    "ecm_remodelling": [
        "SPP1","SPARC","FN1","THBS1",
        "COL1A1","COL1A2","COL3A1","COL4A2","COL8A2",
        "MMP2","MMP9","MMP19",
        "ADAM8","ADAM9","ADAM17",
        "SERPINE1"
    ],

    # =========================
    # ENDOTHELIAL REGULATION
    # =========================

    "endothelial_regulation_program": [
        "VEGFA","VEGFB","FLT1","ENG","VASH1","VEZF1",
        "NRP1","NRP2","UNC5B","ROBO1","PLXDC1",
        "HBEGF","GAS6","AXL","LRG1","ADM"
    ],

    # =========================
    # CARDIOMYOCYTE REGULATION
    # =========================

    "cardiomyocyte_regulation_program": [
        "TNF","IL1B","IL6","IL10",
        "TGFB1","SPP1","HMOX1","NFE2L2",
        "CXCL8","CCL2","ADAM17","SERPINE1",
        "APOE","PPARG"
    ],

    # =========================
    # METABOLISM
    # =========================

    "lipid_handling": [
        "APOE","APOC1","ABCA1","LIPA","PPARG","TSPO","GSTO1"
    ],

    "glycolysis_hypoxia": [
        "HK2","PFKP","PGK1","ALDOA","EGLN1","HIF1A"
    ],

    "oxphos_mitochondria": [
        "SDHB","SDHC","SDHD",
        "UQCRB","UQCRC1","UQCRFS1","UQCRH","UQCRQ","UQCR10","UQCR11",
        "COX4I1","COX5A","COX5B","COX6A1","COX6B1","COX6C",
        "COX7A2","COX7A2L","COX7B","COX7C",
        "CYCS","CYC1","ISCU","SURF1","UQCC2","DGUOK"
    ],

    # =========================
    # STRESS RESPONSE
    # =========================

    "stress_response": [
        "HSP90AA1","HSP90AB1","HSPA8","HSPA1A","HSPA1B",
        "HSPB1","HSPH1",
        "XBP1","ERN1","EIF2AK3",
        "HMOX1","NFE2L2","SOD2",
        "DDIT3","DDIT4","MCL1","CDKN1A","PPIF"
    ],

    # =========================
    # CELL CYCLE
    # =========================

    "cell_cycle_proliferation": [
        "MKI67","TOP2A",
        "PLK1","TTK","AURKA","AURKB",
        "CDC20","CDC6","CDC45","CDK1",
        "MCM2","MCM3","MCM4","MCM5","MCM6","MCM7"
    ]
}


In [ ]:
sn_signatures = {

    "leukocyte_activation_regulation": [
        "FCGR2B","FCGR3A","FCER1G","TYROBP",
        "ITGAM","ITGB2","PTPN6","BTK","HCK",
        "NFKB1","NFKBIA","NFKBIZ","RELB","TNFAIP3",
        "TRAF3","IRAK1","IRAK2","RIPK2",
        "CLEC4E","CR1","C3AR1"
    ],

    "innate_sensing_inflammasome": [
        "TLR4","TLR5","TLR8","LY96","UNC93B1",
        "NLRP3","PYCARD","CARD8","NEK7"
    ],

    "antigen_presentation": [
        "CD74","B2M",
        "HLA-DRA","HLA-DRB1","HLA-DPA1","HLA-DPB1",
        "HLA-DQA1","HLA-DQB1","HLA-DMA","HLA-DMB",
        "CD40","CD80","CD83","CD58"
    ],

    "cytokine_secretion": [
        "IL1B","IL6","TNF","IL10",
        "CCL2","CCL5","CCL8","CCL13","CCL18",
        "CXCL1","CXCL2","CXCL3","CXCL8",
        "PTGS2","ADAM17","OSM"
    ],

    "ecm_remodelling": [
        "SPARC","FN1","THBS1",
        "MMP2","MMP9",
        "ITGA5","ITGB1","ITGB8","ITGAV",
        "PDGFA","PDGFB","ADAM8"
    ],

    "endocytosis_phagocytosis_clearance": [
        "AP2A2","AP2M1","AP2S1","PICALM","EPS15","FCHO2",
        "DNM1","DNM2","RAB31","PIKFYVE","SNX9",
        "LRP1","SCARB2","CD63",
        "MRC1","STAB1","SIGLEC1"
    ],

    "endothelial_regulation_program": [
        "VEGFA","VEGFB","FLT1","ENG","LRG1","ADM",
        "ROBO1","UNC5B","HBEGF","EREG","SPHK1"
    ],
      "cardiomyocyte_regulation_program": [
        "TNF","IL1B","IL6","IL10",
        "TGFB1","SPP1","HMOX1","NFE2L2",
        "CXCL8","CCL2","ADAM17","SERPINE1",
        "APOE","PPARG"
    ],


    "lipid_handling": [
        "APOE","PPARG","ABCA1","LIPA",
        "TREM2","VSIG4","GPNMB","MARCO","LGALS3"
    ],

    "glycolysis_hypoxia": [
        "HK2","ALDOA","PKM","PGK1","HIF1A"
    ],

    "stress_response": [
        "HIF1A","NFE2L2","HMOX1","SOD2",
        "DDIT3","DDIT4",
        "HSPA1A","HSPA1B","HSP90AA1","HSPB1"
    ],

    "cell_cycle_proliferation": [
        "MKI67","TOP2A","PLK1","AURKA","AURKB",
        "CDC20","CDK1","MCM2","MCM3","MCM4","MCM5","MCM6"
    ]
}


In [ ]:
import pyucell as uc

In [ ]:
uc.compute_ucell_scores(adata, signatures=sc_signatures)

In [ ]:
uc.compute_ucell_scores(adata1, signatures=sn_signatures)

# sc0.4 ; sn0.5

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, 
          color = [
                  'LYVE1', 'C1QA', 'NRP1', 'leiden_res0_4',  'CD68',
                  'C1QC', 'MRC1', 'SELENOP', 'F13A1', 'FOLR2'], size = 30,
           color_map = 'RdBu_r', ncols = 2
          )

In [ ]:
sc.pl.umap(adata, 
          color = ['antigen_presentation_UCell', 'ecm_remodelling_UCell', 
                   'endocytosis_phagocytosis_clearance_UCell'
                   ,'endothelial_regulation_program_UCell', 'cardiomyocyte_regulation_program_UCell',
                  'LYVE1', 'C1QA', 'NRP1', 'leiden_res0_4',  'cell_cycle_proliferation_UCell',
                  'S_score', 'G2M_score'], size = 30,
           color_map = 'RdBu_r', ncols = 2
          )


In [ ]:
adata.obs["anatomical_site"] = adata.obs["anatomical_site"].astype("category")

pal_map = {
    "aortic valve": "#020bf7",
    "apex": "#fcca5d",
    "base": "#00ff08",
    "infarct region": "#ff00f2",
    "interventricular septum": "#a200ff",
    "left atrium": "#ededab",
    "left ventricle": "#000000",
    "right atrium": "#ffff00",
    "right ventricle": "#90bd8c",
}

cats = adata.obs["anatomical_site"].cat.categories
sc.pl.umap(adata, color="anatomical_site", palette=[pal_map[c] for c in cats], legend_loc="right margin", size  = 30 )

In [ ]:
adata.obs["Disease"] = adata.obs["Disease"].astype("category")

pal_map =  {
    "healthy": "#009E73",
    "fibrosis": "#E69F00",
    "fibrocalcification": "#56B4E9",
    "heart failure": "#D55E00",
    "ischaemic cardiomyopathy": "#0072B2",
    "myocardial infarction": "#CC79A7",
    "non-ischaemic cardiomyopathy": "#999999",
}


cats = adata.obs["Disease"].cat.categories
sc.pl.umap(adata, color="Disease", palette=[pal_map[c] for c in cats], legend_loc="right margin", size  = 30 )

In [ ]:
sc.pl.umap(adata, 
          color = ['Disease', 'age' ], size = 30,
           color_map = 'RdBu_r', ncols = 1
          )

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['anatomical_site', 'leiden_res0_4']]
frequency_df = obs_df.groupby(['anatomical_site', 'leiden_res0_4']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across anatomical_site / leiden")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['anatomical_site', 'Disease']]
frequency_df = obs_df.groupby(['anatomical_site', 'Disease']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across Disease / anatomical sites")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
cats = adata.obs["Disease"].cat.categories
sc.pl.umap(adata, color="anatomical_site", palette=[pal_map[c] for c in cats], legend_loc="right margin", size  = 30 )

In [ ]:
import matplotlib.pyplot as plt

# 1) define a disease palette (colorblind friendly-ish)
disease_colors = {
    "healthy": "#009E73",
    "fibrosis": "#E69F00",
    "fibrocalcification": "#56B4E9",
    "heart failure": "#D55E00",
    "ischaemic cardiomyopathy": "#0072B2",
    "myocardial infarction": "#CC79A7",
    "non-ischaemic cardiomyopathy": "#999999",
}

# 2) set category order (optional but recommended)
disease_order = [d for d in disease_colors if d in adata.obs["Disease"].unique()]
adata.obs["Disease"] = adata.obs["Disease"].astype("category")
adata.obs["Disease"] = adata.obs["Disease"].cat.reorder_categories(disease_order, ordered=True)

# --- YOUR PLOT (percent stacked barh) ---
obs_df = adata.obs[["anatomical_site", "Disease"]]
frequency_df = obs_df.groupby(["anatomical_site", "Disease"]).size().unstack(fill_value=0)

# keep columns in disease_order and build the palette list in the same order
frequency_df = frequency_df.reindex(columns=disease_order, fill_value=0)
pal_list = [disease_colors[d] for d in disease_order]

percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 3))
percentage_df.plot(kind="barh", stacked=True, ax=ax, color=pal_list)

ax.set_title("Proportion of clusters across Disease / anatomical sites")
ax.set_xlabel("Percentage (%)")
ax.set_ylabel("sample")
ax.legend(title="Disease", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(False)
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
adata1

In [ ]:
sc.pl.umap(adata1, 
          color = ['antigen_presentation_UCell', 'ecm_remodelling_UCell', 
                   'endocytosis_phagocytosis_clearance_UCell'
                   ,'endothelial_regulation_program_UCell', 'cardiomyocyte_regulation_program_UCell',
                  'LYVE1', 'C1QA', 'NRP1', 'leiden_res0_5', 'anatomical_site', 
                  'S_score', 'G2M_score'], size = 30,
           color_map = 'RdBu_r', ncols = 2
          )


In [ ]:
sc.pl.umap(adata1, 
          color = [ 'disease'
                  , 'age_group'], size = 30,
           color_map = 'RdBu_r', ncols = 1
          )


In [ ]:
sc.pl.umap(adata1, 
          color = ['anatomical_site'], size = 10,
           
          )


In [ ]:
import decoupler as dc

In [ ]:
collectri = dc.op.collectri(organism='human')
collectri

In [ ]:
dc.mt.ulm(data=adata, net=collectri)

In [ ]:
dc.mt.ulm(data=adata1, net=collectri)

In [ ]:
score = dc.pp.get_obsm(adata=adata, key='score_ulm')
score

In [ ]:
score = dc.pp.get_obsm(adata=adata1, key='score_ulm')
score

In [ ]:
df = dc.tl.rankby_group(
    adata=score,
    groupby='leiden_res0_4',
    reference='rest',
    method='t-test_overestim_var'
)
df = df[(df['stat'] > 0) & (df['padj'] < 0.01)]
df

In [ ]:
df = dc.tl.rankby_group(
    adata=score,
    groupby='leiden_res0_5',
    reference='rest',
    method='t-test_overestim_var'
)
df = df[(df['stat'] > 0) & (df['padj'] < 0.01)]
df

In [ ]:
n_markers = 5
source_markers = (
    df
    .groupby('group')
    .head(n_markers)
    .drop_duplicates('name')
    .groupby('group')
    ['name']
    .apply(lambda x: list(x))
    .to_dict()
)

In [ ]:
#sc.tl.dendrogram(score,   groupby='leiden_res0_4')
sc.tl.dendrogram(score,   groupby='leiden_res0_5')

In [ ]:
sc.pl.dotplot(adata,
            var_names = ['LYVE1'], groupby = 'leiden_res0_4')

In [ ]:
sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby='leiden_res0_4',
   dendrogram=True,
    standard_scale= 'var' ,
    colorbar_title='Z-scaled scores',
    cmap='RdBu_r',
    #save = 'dc_day5'
)

In [ ]:
sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby='leiden_res0_5',
   dendrogram=True,
    standard_scale= 'var' ,
    colorbar_title='Z-scaled scores',
    cmap='RdBu_r',
    #save = 'dc_day5'
)

In [ ]:
sc.pl.violin(adata1, )